# Anhedonic AI — Activation Visualization
Explore MLP intermediate activations extracted from Qwen2-VL-7B across 6 conditions (geo/math × neutral/reward/money).

**Sections:**
1. Load & inspect data
2. Mean activation magnitude per layer
3. Mean |delta| per layer — where is the incentive signal?
4. Delta heatmaps — spatial structure across layers & neurons
5. 3σ threshold counts per layer
6. Delta distributions — is the signal real?
7. Geo vs Math correlation — are universal neurons truly universal?
8. Per-question variance — is the signal stable?
9. Summary statistics & go/no-go for extraction

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 10

ACTIVATIONS_DIR = 'activations'
DOMAINS    = ['geo', 'math']
CONDITIONS = ['neutral', 'reward', 'money']
COLORS     = {'neutral': '#607d8b', 'reward': '#e91e63', 'money': '#4caf50'}
DLABELS    = {'geo': 'Geography', 'math': 'Math'}

print('Libraries loaded.')

## 1 · Load & Inspect Data

In [ ]:
# acts[domain][condition] = np.array [28, 18944]  — mean across 100 questions
# raw[domain][condition]  = np.array [100, 28, 18944]  — per-question (for variance)

acts = {}
raw  = {}

for domain in DOMAINS:
    acts[domain] = {}
    raw[domain]  = {}
    for cond in CONDITIONS:
        path = os.path.join(ACTIVATIONS_DIR, f'{cond}_activations_{domain}.pt')
        data = torch.load(path, map_location='cpu')
        tensors = torch.stack(list(data.values())).float()   # [100, 28, 18944]
        raw[domain][cond]  = tensors.numpy()
        acts[domain][cond] = tensors.mean(dim=0).numpy()     # [28, 18944]
        print(f'  {domain}/{cond:8s}  shape={acts[domain][cond].shape}  '
              f'mean={acts[domain][cond].mean():.5f}  std={acts[domain][cond].std():.5f}')

num_layers       = acts['geo']['neutral'].shape[0]
intermediate_dim = acts['geo']['neutral'].shape[1]
layers           = np.arange(num_layers)

print(f'\nLayers: {num_layers}  |  MLP intermediate dim: {intermediate_dim}')
print(f'Total neurons in network: {num_layers * intermediate_dim:,}')

In [ ]:
# Compute deltas — reused in all subsequent plots
# deltas[domain][condition] = condition_acts - neutral_acts  shape: [28, 18944]

deltas = {}
for domain in DOMAINS:
    deltas[domain] = {
        'reward': acts[domain]['reward'] - acts[domain]['neutral'],
        'money':  acts[domain]['money']  - acts[domain]['neutral'],
    }

for domain in DOMAINS:
    for cond in ['reward', 'money']:
        d = deltas[domain][cond]
        print(f'  delta {domain}/{cond:8s}  mean={d.mean():.6f}  std={d.std():.6f}  max|d|={np.abs(d).max():.6f}')

## 2 · Mean Activation Magnitude per Layer
Are all three conditions in the same ballpark? Does the model do more computation in specific layers?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
fig.suptitle('Mean |Activation| per Layer by Condition', fontweight='bold')

for ax, domain in zip(axes, DOMAINS):
    for cond in CONDITIONS:
        mag = np.abs(acts[domain][cond]).mean(axis=1)
        ax.plot(layers, mag, color=COLORS[cond], label=cond.capitalize(),
                linewidth=2, marker='o', markersize=3)
    ax.set_title(DLABELS[domain])
    ax.set_xlabel('Layer')
    ax.set_ylabel('Mean |Activation|')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, num_layers, 2))

plt.tight_layout()
plt.show()

## 3 · Mean |Delta| per Layer
Where in the network does the incentive prefix create the biggest change?  
Clear peaks = the signal is localised. Flat line = signal is too weak or diffuse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
fig.suptitle('Mean |Delta| per Layer  (condition − neutral)', fontweight='bold')

for ax, domain in zip(axes, DOMAINS):
    for cond, color in [('reward', COLORS['reward']), ('money', COLORS['money'])]:
        d = np.abs(deltas[domain][cond]).mean(axis=1)
        ax.plot(layers, d, color=color, label=f'{cond.capitalize()} − Neutral',
                linewidth=2, marker='o', markersize=3)
    ax.set_title(DLABELS[domain])
    ax.set_xlabel('Layer')
    ax.set_ylabel('Mean |Delta|')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, num_layers, 2))

plt.tight_layout()
plt.show()

## 4 · Delta Heatmaps — Spatial Structure
Neurons binned into 64 groups for readability.  
**Red** = more active with incentive, **Blue** = suppressed by incentive.

In [ ]:
BIN_SIZE = intermediate_dim // 64
n_bins   = intermediate_dim // BIN_SIZE

fig, axes = plt.subplots(2, 2, figsize=(18, 9))
fig.suptitle('Delta Heatmap (layers × neuron bins)\nRed = activated by incentive  |  Blue = suppressed',
             fontweight='bold')

for (r, c, domain, cond, title) in [
    (0, 0, 'geo',  'reward', 'Geography — Reward − Neutral'),
    (0, 1, 'geo',  'money',  'Geography — Money − Neutral'),
    (1, 0, 'math', 'reward', 'Math — Reward − Neutral'),
    (1, 1, 'math', 'money',  'Math — Money − Neutral'),
]:
    ax = axes[r][c]
    d  = deltas[domain][cond]
    d_binned = d[:, :n_bins * BIN_SIZE].reshape(num_layers, n_bins, BIN_SIZE).mean(axis=2)
    vmax = np.percentile(np.abs(d_binned), 98)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im = ax.imshow(d_binned, aspect='auto', cmap='RdBu_r', norm=norm, origin='upper')
    plt.colorbar(im, ax=ax, fraction=0.03)
    ax.set_title(title)
    ax.set_xlabel(f'Neuron bin  (each ≈ {BIN_SIZE} neurons)')
    ax.set_ylabel('Layer')
    ax.set_yticks(range(0, num_layers, 2))
    ax.set_yticklabels(range(0, num_layers, 2))

plt.tight_layout()
plt.show()

## 5 · 3σ Threshold — How Many Neurons Cross It?
**Top row:** significant neurons per layer, per domain.  
**Bottom row:** cross-domain overlap (universal neurons) — what `extract_neurons.py` will use.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle('Neurons Exceeding 3σ Threshold per Layer', fontweight='bold')

for col, cond in enumerate(['reward', 'money']):
    thr_g = 3 * np.std(deltas['geo'][cond])
    thr_m = 3 * np.std(deltas['math'][cond])

    geo_counts, math_counts, overlap_counts = [], [], []
    for layer in range(num_layers):
        sig_g = set(np.where(np.abs(deltas['geo'][cond][layer])  > thr_g)[0])
        sig_m = set(np.where(np.abs(deltas['math'][cond][layer]) > thr_m)[0])
        geo_counts.append(len(sig_g))
        math_counts.append(len(sig_m))
        overlap_counts.append(len(sig_g & sig_m))

    ax_top = axes[0][col]
    ax_bot = axes[1][col]

    ax_top.bar(layers - 0.2, geo_counts,  0.4, label='Geo',  color='#2196f3', alpha=0.8)
    ax_top.bar(layers + 0.2, math_counts, 0.4, label='Math', color='#ff9800', alpha=0.8)
    ax_top.set_title(f'{cond.capitalize()} − Neutral: significant neurons per layer')
    ax_top.set_xlabel('Layer')
    ax_top.set_ylabel('# neurons > 3σ')
    ax_top.legend()
    ax_top.grid(True, alpha=0.3)
    ax_top.set_xticks(range(0, num_layers, 2))

    ax_bot.bar(layers, overlap_counts, color='#9c27b0', alpha=0.85)
    ax_bot.set_title(f'{cond.capitalize()}: cross-domain overlap (universal) per layer')
    ax_bot.set_xlabel(f'Layer   (total universal neurons: {sum(overlap_counts):,})')
    ax_bot.set_ylabel('# overlapping neurons')
    ax_bot.grid(True, alpha=0.3)
    ax_bot.set_xticks(range(0, num_layers, 2))

plt.tight_layout()
plt.show()

## 6 · Delta Distributions
Should be zero-centred with roughly Gaussian shape.  
The **% outside 3σ** tells you if the signal is real — by chance it should be ~0.27%.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle('Delta Distributions — is the signal real?', fontweight='bold')

for row, domain in enumerate(DOMAINS):
    for col, cond in enumerate(['reward', 'money']):
        ax  = axes[row][col]
        d   = deltas[domain][cond].flatten()
        std = np.std(d)
        pct = np.mean(np.abs(d) > 3 * std) * 100
        ax.hist(d, bins=400, color=COLORS[cond], alpha=0.75, density=True)
        ax.axvline( 3*std, color='black', linestyle='--', linewidth=1.2, label=f'+3σ = {3*std:.4f}')
        ax.axvline(-3*std, color='black', linestyle='--', linewidth=1.2, label=f'-3σ = {-3*std:.4f}')
        ax.set_title(f'{DLABELS[domain]} — {cond.capitalize()} − Neutral\n'
                     f'σ = {std:.5f}   |   {pct:.3f}% > 3σ   (chance = 0.270%)')
        ax.set_xlabel('Delta value')
        ax.set_ylabel('Density')
        ax.set_xlim(np.percentile(d, 0.05), np.percentile(d, 99.95))
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7 · Geo vs Math Delta Correlation
Each point = one (layer, neuron) pair. **Coloured points** crossed 3σ in both domains = universal neurons.  
High Pearson r + tight coloured clusters in the corners = strong cross-domain universality.

In [ ]:
np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Geo vs Math Delta Correlation\nColoured = universal neurons (> 3σ in both domains)', fontweight='bold')

for ax, cond in zip(axes, ['reward', 'money']):
    dg = deltas['geo'][cond].flatten()
    dm = deltas['math'][cond].flatten()
    thr_g = 3 * np.std(deltas['geo'][cond])
    thr_m = 3 * np.std(deltas['math'][cond])

    idx = np.random.choice(len(dg), size=min(60000, len(dg)), replace=False)
    dg_s, dm_s = dg[idx], dm[idx]
    univ = (np.abs(dg_s) > thr_g) & (np.abs(dm_s) > thr_m)

    ax.scatter(dg_s[~univ], dm_s[~univ], s=0.3, alpha=0.15, color='#90a4ae', rasterized=True)
    ax.scatter(dg_s[univ],  dm_s[univ],  s=5,   alpha=0.9,  color=COLORS[cond],
               label=f'Universal ({univ.sum():,} in sample)')
    for sign in [1, -1]:
        ax.axvline(sign * thr_g, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.axhline(sign * thr_m, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

    corr = np.corrcoef(dg_s, dm_s)[0, 1]
    ax.set_title(f'{cond.capitalize()} − Neutral   |   Pearson r = {corr:.4f}')
    ax.set_xlabel('Geo delta')
    ax.set_ylabel('Math delta')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 8 · Per-Question Variance Check
Low variance across questions = the same neurons activate consistently regardless of which question is asked.  
This is the stability check — unstable neurons are noise, not signal.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Per-Question Delta Variance per Layer\nLow = signal is stable across questions', fontweight='bold')

for row, domain in enumerate(DOMAINS):
    for col, cond in enumerate(['reward', 'money']):
        ax = axes[row][col]
        # raw shape: [100, 28, 18944]
        per_q_delta = raw[domain][cond] - raw[domain]['neutral']  # [100, 28, 18944]
        var_per_layer = per_q_delta.var(axis=0).mean(axis=1)       # [28]
        ax.plot(layers, var_per_layer, color=COLORS[cond], linewidth=2, marker='o', markersize=3)
        ax.set_title(f'{DLABELS[domain]} — {cond.capitalize()} − Neutral')
        ax.set_xlabel('Layer')
        ax.set_ylabel('Variance across questions')
        ax.grid(True, alpha=0.3)
        ax.set_xticks(range(0, num_layers, 2))

plt.tight_layout()
plt.show()

## 9 · Summary Statistics & Go/No-Go for Extraction

In [ ]:
print('=' * 65)
print('SUMMARY STATISTICS')
print('=' * 65)

for domain in DOMAINS:
    print(f'\n{DLABELS[domain]}:')
    for cond in ['reward', 'money']:
        d   = deltas[domain][cond]
        std = np.std(d)
        n_sig = int(np.sum(np.abs(d) > 3 * std))
        pct   = n_sig / (num_layers * intermediate_dim) * 100
        print(f'  {cond.capitalize()} − Neutral:')
        print(f'    mean delta   = {d.mean():.7f}')
        print(f'    std  delta   = {std:.7f}')
        print(f'    max |delta|  = {np.abs(d).max():.7f}')
        print(f'    neurons > 3σ = {n_sig:,}  ({pct:.3f}%  |  chance baseline = 0.270%)')

print(f'\n{"-" * 65}')
print('Cross-domain universal neurons (geo ∩ math):')
print(f'{"-" * 65}')

univ = {}
for cond in ['reward', 'money']:
    thr_g = 3 * np.std(deltas['geo'][cond])
    thr_m = 3 * np.std(deltas['math'][cond])
    sig_g = set(map(tuple, np.argwhere(np.abs(deltas['geo'][cond])  > thr_g).tolist()))
    sig_m = set(map(tuple, np.argwhere(np.abs(deltas['math'][cond]) > thr_m).tolist()))
    univ[cond] = sig_g & sig_m
    print(f'  {cond.capitalize():8s}: {len(sig_g):,} geo  ∩  {len(sig_m):,} math  =  {len(univ[cond]):,} universal')

master_core = univ['reward'] & univ['money']
print(f'\n{"-" * 65}')
print(f'Master core (reward_universal ∩ money_universal): {len(master_core):,} neurons')
print(f'{"-" * 65}')

if master_core:
    core_layers = [l for l, n in master_core]
    print('Layer distribution:')
    for layer in sorted(set(core_layers)):
        count = core_layers.count(layer)
        bar   = '█' * min(count, 50)
        print(f'  Layer {layer:>2}: {count:>4}  {bar}')
    print(f'\n✓  Ready to run extract_neurons.py')
else:
    print('\n✗  Master core is empty — consider lowering threshold or revisiting prompt design')